# Evaluation (development period, 2016-2023)

Runs the tests fixed in `docs/preregistration.md`.
The holdout (2024-2026) is not touched here; it lives in `04_holdout.ipynb` and is run once at the end.

Sign convention: Thesis A (informed shorting) predicts a **negative** IC, meaning high short volume leads to underperformance.

In [1]:
import numpy as np
import pandas as pd

from shortvol import evaluate as ev, features, finra, prices

pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)

panel = pd.read_parquet(prices.ROOT / "data" / "processed" / "panel.parquet")
days = finra.trading_days("2016-01-01", "2026-08-31")
V0 = features.estimate_v0(features.add_all(panel, days))["v0"]  # 2016-2019 short volume only
df = features.add_all(panel, days, v0=V0)

SIGNALS = ["svr5", "svr1", "svr_abnormal", "svr5_shrunk"]
CONTROLS = ["ret1", "ret5", "log_adv20", "abnormal_volume"]
TARGETS = ["fwd_oc", "fwd_cc"]
for s in SIGNALS:
    df[s + "_n"] = ev.neutralize(df, s, CONTROLS)
df["size_tercile"] = ev.terciles(df, "log_adv20", ["small", "mid", "large"])
df["offex_tercile"] = ev.terciles(df, "off_exchange_share", ["low", "mid", "high"])

In [2]:
d = df[df.date <= "2023-12-29"]
print(f"development stock-days: {len(d):,}, days: {d.date.nunique()}")

development stock-days: 985,579, days: 2012


## 1. Headline: 4 signals x 2 targets, raw and neutralized
Raw IC is the signal as is.
Neutralized IC removes day-t return, 5-day return, size (20-day dollar volume), abnormal volume and industry.
The gap between them shows how much of the signal is reversal, size or liquidity in disguise.

In [3]:
rows = []
for s in SIGNALS:
    for t in TARGETS:
        for kind, col in [("raw", s), ("neutralized", s + "_n")]:
            rows.append({"signal": s, "target": t, "version": kind, **ev.summarize_ic(ev.daily_ic(d, col, t))})
headline = pd.DataFrame(rows).set_index(["signal", "target", "version"])
headline

mean_ic    nw_t  ic_ir_annual  pct_positive  days
signal       target version                                                       
svr5         fwd_oc raw           0.0022  1.3151        0.4854        0.5020  2008
                    neutralized   0.0007  0.4933        0.1844        0.5050  1998
             fwd_cc raw          -0.0009 -0.5634       -0.2032        0.4846  2008
                    neutralized  -0.0014 -1.0413       -0.3822        0.4855  1998
svr1         fwd_oc raw          -0.0035 -2.1925       -0.8158        0.4881  2012
                    neutralized  -0.0043 -3.2603       -1.2216        0.4805  1998
             fwd_cc raw          -0.0007 -0.4817       -0.1741        0.4896  2012
                    neutralized  -0.0006 -0.5148       -0.1869        0.4940  1998
svr_abnormal fwd_oc raw           0.0012  0.9594        0.3470        0.5084  1967
                    neutralized  -0.0002 -0.2164       -0.0800        0.4982  1967
             fwd_cc raw           0.0010  0.8425        0.2939        0.5114  1967
                    neutralized  -0.0001 -0.1002       -0.0350        0.4962  1967
svr5_shrunk  fwd_oc raw           0.0022  1.2791        0.4775        0.5013  1973
                    neutralized   0.0007  0.4892        0.1842        0.5079  1973
             fwd_cc raw          -0.0010 -0.6240       -0.2268        0.4901  1973
                    neutralized  -0.0016 -1.1850       -0.4372        0.4876  1973

## 2. Quintile long-short (primary target)
Long the top quintile of the signal, short the bottom quintile, equal weight, rebalanced daily.
Turnover is the daily sum of absolute weight changes (a full flip of both legs is 4).
Break-even is the one-way cost in bps that erases the gross spread. It is reported, not pass/fail.

In [4]:
pd.DataFrame({f"{s}_n": ev.quintile_spread(d, s + "_n", "fwd_oc") for s in SIGNALS}).T

,spread_bps,spread_nw_t,turnover,breakeven_bps
svr5_n,0.2441,0.3558,0.7684,0.3177
svr1_n,-1.7235,-2.6926,2.1121,0.8160
svr_abnormal_n,-0.4089,-0.7907,0.8846,0.4623
svr5_shrunk_n,0.1941,0.2828,0.7584,0.2559


## 3. Decay
IC of the primary neutralized signal against the open-to-close return 1, 2, 5, 10 and 20 days ahead.
Informed trading should decay slowly; price pressure and reversal should die within a day or two.

In [5]:
pd.DataFrame({k: ev.summarize_ic(ev.daily_ic(d.assign(y=ev.shift_target(d, "fwd_oc", k)), "svr5_n", "y"))
              for k in [1, 2, 5, 10, 20]}).T

,mean_ic,nw_t,ic_ir_annual,pct_positive,days
1,0.0007,0.4933,0.1844,0.5050,1998.0000
2,0.0030,2.2062,0.8294,0.5208,1997.0000
5,0.0029,2.2033,0.8111,0.5291,1994.0000
10,0.0020,1.6007,0.5671,0.5098,1989.0000
20,0.0019,1.4886,0.5495,0.5195,1979.0000


## 4. Splits (primary signal, primary target)
By size, by period, and by off-exchange share.
Thesis B predicts the relation weakens or flips where off-exchange share is high.

In [6]:
d = d.assign(period=pd.cut(d.date, pd.to_datetime(["2015-12-31", "2019-12-31", "2021-12-31", "2023-12-31"]),
                           labels=["2016-19", "2020-21", "2022-23"]))
pd.concat({
    "size": ev.ic_by_group(d, "svr5_n", "fwd_oc", "size_tercile"),
    "period": ev.ic_by_group(d, "svr5_n", "fwd_oc", "period"),
    "off-exchange share": ev.ic_by_group(d, "svr5_n", "fwd_oc", "offex_tercile"),
})

mean_ic    nw_t  ic_ir_annual  pct_positive      days
size               large     0.0003  0.1322        0.0471        0.5075 1998.0000
                   mid       0.0006  0.3064        0.1119        0.4990 1998.0000
                   small     0.0013  0.6472        0.2375        0.5130 1998.0000
period             2016-19   0.0007  0.3704        0.1974        0.4960  992.0000
                   2020-21   0.0005  0.1876        0.1421        0.5149  505.0000
                   2022-23   0.0008  0.2751        0.2013        0.5130  501.0000
off-exchange share high     -0.0045 -2.1528       -0.7716        0.4845 1998.0000
                   low       0.0027  1.4917        0.5317        0.5090 1998.0000
                   mid       0.0037  1.8698        0.6843        0.5195 1998.0000

## 5. Success bar (development part)
1. Mean IC negative with |NW t| >= 3.
2. The same sign in at least 2 of the 3 development periods.
Criterion 3 (holdout) is checked in `04_holdout.ipynb`.

In [7]:
primary = headline.loc[("svr5", "fwd_oc", "neutralized")]
periods = ev.ic_by_group(d, "svr5_n", "fwd_oc", "period")
c1 = primary.mean_ic < 0 and abs(primary.nw_t) >= 3
c2 = (np.sign(periods.mean_ic) == np.sign(primary.mean_ic)).sum() >= 2
print(f"1. mean IC {primary.mean_ic:.4f}, NW t {primary.nw_t:.2f}: {'PASS' if c1 else 'FAIL'}")
print(f"2. same sign in {(np.sign(periods.mean_ic) == np.sign(primary.mean_ic)).sum()} of 3 periods: {'PASS' if c2 else 'FAIL'}")

1. mean IC 0.0007, NW t 0.49: FAIL
2. same sign in 3 of 3 periods: PASS
